In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time
from utils import *


In [ ]:
# User-Agent atualizado
ARQUIVO = "decks_fabtcg.csv"

session = requests.Session()
session.headers.update(headers)

In [ ]:
def classificar_evento(nome_evento):
    nome_lower = str(nome_evento).lower()
    for chave, valor in EVENT_MAPPING.items():
        if chave in nome_lower:
            return valor
    return "outro"

def extrair_dados():
    dados = []
    page = 1

    while True:
        url = f'https://fabtcg.com/decklists/page/{page}/'
        response = session.get(url)
        if response.status_code != 200:
            print(f"Erro HTTP {response.status_code} na página {page}. Interrompendo.")
            break
            
        print(f"<============= Escaneando a página {page} =============>")
        soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find('table')
        
        if table:
            linhas = table.find_all('tr')[1:]

            for linha in linhas:
                colunas = linha.find_all('td')

                dado = [coluna.text.strip() for coluna in colunas]

                coluna_deck = colunas[2] 
                elemento_a = coluna_deck.find('a')
                if elemento_a:
                    link_do_deck = elemento_a.get('href')
                    dado.append(link_do_deck)
                else:
                    dado.append("Sem link")

                if len(dado[3]) < 1 : continue

                tipo_evento = classificar_evento(dado[3])
                dado.append(tipo_evento)
                dados.append(dado)
        else:
            print(f"Erro, nehuma table encontrada na pagina: {page}")

        page +=1
        time.sleep(0.5)


    return dados


In [ ]:

if arquivo_e_atual(ARQUIVO):
    print("O arquivo esta atualizado.")
    
else:
    
    apagar_arquivo(ARQUIVO)
    apagar_arquivo("decks_fabtcg.parquet")

    #print("Iniciando a extração.")
    
    dados = extrair_dados()

    colunas_df = ["Pais", "Data", "Decklist", "Evento", "Formato", "Heroi", "Resultado","Link do deck", "Tipo de evento"]
    df = pd.DataFrame(dados, columns=colunas_df)

    df['Resultado'] = df['Resultado'].replace('', 'nao informado')

    df.to_csv("decks_fabtcg.csv", index=False)
    df.to_parquet("decks_fabtcg.parquet", index=False)
    #print("Dados extraidos e salvos localmente!")


